# Part 1B — Regression ML Pipeline

**Question:** can the measured baseline variables in scikit-learn's bundled diabetes dataset estimate its quantitative disease-progression score?

**Regression** predicts a continuous number rather than a category. A **residual** is the actual value minus the predicted value.

> **Teaching boundary:** this is an educational benchmark. It is **not** a medical forecast, a treatment recommendation, or a deployment model.

**Roadmap:** load → inspect → split → pipeline → cross-validate → evaluate once → residual diagnostics → leakage demonstration → interpret limits.

## 0. Before you begin

**Runtime:** about 5 minutes of computation. Section 7 adds roughly 30 seconds.

**Data:** `load_diabetes()` ships with scikit-learn. No upload, no download.

**If anything breaks:** restart the kernel and use **Run All**.

In [ ]:
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.datasets import load_diabetes
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()} | scikit-learn: {sklearn.__version__}")

## 1. Load and inspect the bundled dataset

The target is a quantitative measure of disease progression one year after baseline. It is a number on an arbitrary scale, not a diagnosis and not a probability.

**Look for:** the centre and spread of the target, because every error metric below is expressed in these units.

Documentation: [`load_diabetes`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html).

In [ ]:
dataset = load_diabetes(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()

print(f"Features: {X.shape[1]} | observations: {X.shape[0]}")
display(X.head())
display(y.describe().to_frame("target"))

### Plot the target distribution

**Look for:** whether the target is roughly symmetric or skewed, and where most of the values sit. Error metrics below are reported in these units, so the spread here sets the scale for judging them.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(y, bins=25, kde=True, ax=ax)
ax.set_title("Diabetes dataset: target distribution")
ax.set_xlabel("Quantitative progression target")
plt.tight_layout()
plt.show()

### What do we see?

442 observations and 10 features — a small dataset. The target runs from about 25 to about 346, with a mean near 152.

Keep that spread in mind. A standard deviation of roughly 77 means that simply predicting the mean every time would give typical errors in the region of 60–70. Any model has to beat that to be worth anything, and section 4 checks exactly this.

The features are already standardised by scikit-learn, so their values look small and centred on zero. We still keep a scaler in the pipeline, for reasons section 3 explains.

## 2. Split before fitting anything

Reserve 20% as the held-out test set. There is no `stratify` argument here because the target is continuous, not categorical.

> **Do not use the test set to choose anything.** It is evaluated once, after model selection is complete.

**Look for:** similar summary statistics in both parts. Large differences would mean the split was unlucky.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Training rows: {len(X_train)} | test rows: {len(X_test)}")
display(pd.DataFrame({"training": y_train.describe(), "test": y_test.describe()}).round(2))

### What do we see?

The two parts have similar means and spreads, so the split looks representative.

With only 442 rows this is worth checking. On a small dataset a single unlucky split can move the final number noticeably, which is one reason the test estimate below should be read as approximate rather than precise.

## 3. Build pipelines

The **dummy regressor** predicts the training mean for every row, ignoring all features. It is the reference any real model must beat.

**Ridge** is linear regression with an L2 penalty on the coefficients, which discourages any single coefficient from becoming large.

Scaling matters specifically *because* of that penalty. The penalty is applied to coefficient size, and coefficient size depends on the units of the feature. Without scaling, a variable measured in large units would be penalised differently from one measured in small units for no principled reason.

Documentation: [`Ridge`](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html).

> ### Your turn
>
> The next code cell is incomplete. Replace every `# TODO` with working code.
> The surrounding explanation tells you what the cell must do and what to look for.
> If you get stuck, the reasoning you need is in the Markdown directly above.

In [ ]:
# TODO: build the preprocessing pipeline (impute with median, then standardise).
preprocessing = Pipeline([
    # TODO
])

# TODO: baseline pipeline using DummyRegressor(strategy="mean")
dummy_pipeline = Pipeline([
    # TODO
])

# TODO: candidate pipeline using Ridge(alpha=1.0)
regression_pipeline = Pipeline([
    # TODO
])

print(regression_pipeline)

## 4. Compare workflows with cross-validation on the training data only

Three metrics, each answering a different question:

- **MAE** — the typical absolute error, in target units. Easy to explain.
- **RMSE** — also in target units, but squares errors first, so large mistakes count for more.
- **R²** — the proportion of variance explained compared with predicting the mean. 0 means no better than the mean; negative means worse.

> **A scikit-learn convention that confuses everyone:** the library treats higher as better for every scorer, so error metrics are supplied *negated* as `neg_mean_absolute_error`. The code multiplies them by −1 to display them as positive errors.

**Look for:** whether Ridge beats the dummy, and by how much in target units.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {"mae": "neg_mean_absolute_error",
           "rmse": "neg_root_mean_squared_error",
           "r2": "r2"}

rows = []
for name, pipeline in {"Dummy mean": dummy_pipeline,
                       "Ridge regression": regression_pipeline}.items():
    scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "MAE mean": -scores["test_mae"].mean(), "MAE SD": scores["test_mae"].std(),
        "RMSE mean": -scores["test_rmse"].mean(), "RMSE SD": scores["test_rmse"].std(),
        "R2 mean": scores["test_r2"].mean(), "R2 SD": scores["test_r2"].std(),
    })

cv_results = pd.DataFrame(rows).set_index("model").round(2)
display(cv_results)

### What do we see?

Ridge improves on the dummy baseline, but look at the size of the improvement rather than just its direction. MAE drops by roughly 20 points on a target whose standard deviation is about 77, and R² lands around 0.45.

The dummy's R² is approximately 0 by construction — that is what R² measures against.

**An R² near 0.45 means most of the variation is still unexplained.** That is a normal result for a small clinical dataset with ten features, and reporting it plainly is the correct behaviour. A model can be correctly built, genuinely better than baseline, and still not be good enough to act on.

Notice also that the standard deviations are not tiny. With 442 rows, fold-to-fold variation is real, and small differences between candidate models would not be meaningful.

## 5. Evaluate once on the held-out test set

Fit Ridge on all the training data, then predict the test rows one time. The dummy is fitted and evaluated alongside it for reference.

**Look for:** whether the test numbers are close to the cross-validation estimates.

In [ ]:
regression_pipeline.fit(X_train, y_train)
y_pred = regression_pipeline.predict(X_test)

dummy_pipeline.fit(X_train, y_train)
dummy_pred = dummy_pipeline.predict(X_test)

comparison = pd.DataFrame({
    "Dummy mean": [mean_absolute_error(y_test, dummy_pred),
                   root_mean_squared_error(y_test, dummy_pred),
                   r2_score(y_test, dummy_pred)],
    "Ridge": [mean_absolute_error(y_test, y_pred),
              root_mean_squared_error(y_test, y_pred),
              r2_score(y_test, y_pred)],
}, index=["MAE", "RMSE", "R2"])
display(comparison.round(2))

### What do we see?

The test results are broadly consistent with the cross-validation estimates, which suggests the workflow did not overfit the training folds.

Note the dummy's R² on the test set. It is slightly **negative**, not exactly zero, because the dummy predicts the *training* mean while R² is computed against the *test* mean. Those differ a little.

**This is the single most misread number in regression.** A negative R² does not mean negative correlation. It means the model's squared error is larger than you would get by predicting the mean of the evaluation set. Never take its absolute value.

## 6. Inspect predictions and residuals

Summary metrics hide patterns. Two plots reveal them:

- **Actual versus predicted:** points should sit near the 45-degree line.
- **Residuals versus predicted:** points should scatter evenly around zero with no visible shape.

A **residual** is actual minus predicted, so a positive residual means the model predicted too low.

**Look for:** any trend in the right-hand plot. A tilt means the model is systematically wrong in a way that depends on its own prediction.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
axes[0].scatter(y_test, y_pred, alpha=0.7)
axes[0].plot([low, high], [low, high], "k--")
axes[0].set_title("Held-out test: actual vs predicted")
axes[0].set_xlabel("Actual target")
axes[0].set_ylabel("Predicted target")

axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("Held-out test: residuals vs predicted")
axes[1].set_xlabel("Predicted target")
axes[1].set_ylabel("Residual (actual - predicted)")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} | SD of residuals: {residuals.std():.2f}")
print(f"Correlation between residual and predicted value: {np.corrcoef(y_pred, residuals)[0, 1]:.3f}")
print(f"Spread of predictions (SD): {y_pred.std():.1f}  vs  spread of actual values (SD): {y_test.std():.1f}")

### What do we see?

**The right plot looks the way it should.** The residuals scatter around zero with no obvious funnel or curve, and their correlation with the predicted value is about −0.08 — essentially flat. There is no strong systematic bias to report, and saying so plainly is the correct result. A residual plot that shows nothing is a passed check, not a wasted cell.

**The left plot is where the real finding is.** The points form a broad cloud around the diagonal rather than a tight band, and the cloud is noticeably flatter than the diagonal. The printed numbers confirm it: the predictions have a standard deviation of about 54 while the actual values have about 73.

The model is **compressing its predictions toward the middle**. It rarely predicts very high or very low values, even when the true value is extreme. This is expected behaviour for a penalised linear model on a weakly predictive dataset — the Ridge penalty shrinks coefficients, which shrinks the spread of predictions.

It matters clinically. A model that pulls everyone toward average is least informative exactly where the target is most extreme, which is often where a decision would actually be made. Note that this limitation is invisible in MAE, RMSE, and R² alike. Only the plot shows it.

Residual plots can reveal patterns like this. They do not *prove* that regression assumptions hold.

## 7. Watch preprocessing leakage invent a result from nothing

Part 0 covered leakage from future information. This section demonstrates the other kind: letting a step that should be fitted inside each fold see all the data first.

The demonstration uses **pure random noise for both the features and the target.** There is no relationship to find. The correct answer is R² ≈ 0.

We then select the 20 features that correlate best with the target — a completely ordinary-looking step — and do it in two different places.

**Look for:** what the "wrong way" reports, knowing that the true answer is zero.

> ### Your turn
>
> The next code cell is incomplete. Replace every `# TODO` with working code.
> The surrounding explanation tells you what the cell must do and what to look for.
> If you get stuck, the reasoning you need is in the Markdown directly above.

In [ ]:
# TODO: build the CORRECT version. Put SelectKBest INSIDE the pipeline so it is
# refitted within each training fold, then cross-validate the pipeline itself.
honest_pipeline = Pipeline([
    # TODO
])
honest_scores = cross_val_score(honest_pipeline, X_noise, y_noise, cv=noise_cv, scoring="r2")

display(pd.DataFrame({
    "approach": ["WRONG: selection before cross-validation", "CORRECT: selection inside the pipeline"],
    "mean R2": [leaky_scores.mean().round(3), honest_scores.mean().round(3)],
    "fold R2 values": [np.round(leaky_scores, 3), np.round(honest_scores, 3)],
}))
print("\nThere is no signal in this data. The true R2 is 0.")

### What do we see?

The leaky approach reports a mean R² of about **+0.32** on data containing no signal whatsoever. Every fold looks positive and consistent. Nothing about the output looks broken.

The correct approach reports about **−0.28**, which is the honest answer: slightly worse than predicting the mean, exactly as you would expect from noise.

**How the leak happened.** `SelectKBest` looked at all 200 rows, including their target values, to decide which 20 of 2,000 noise features correlated best. With 2,000 candidates, some will correlate with the target by chance. Those chance correlations were then baked into the data *before* the folds were created, so every validation fold was scored on features that had already been tuned to it.

Three things make this dangerous in practice:

1. **It produces a better-looking number**, so nobody investigates.
2. **The code looks completely reasonable.** Selecting features before modelling is a normal-sounding thing to do.
3. **It survives a held-out test set** if that split happened after the selection step.

**The defence is structural, not vigilance.** Put every step that learns anything from the data — imputers, scalers, encoders, feature selectors — inside the `Pipeline`. Then cross-validation refits them within each training fold and the leak becomes impossible by construction.

See `../handouts/leakage_checklist.md` for the full checklist.

## 8. Try it yourself

1. In section 7, change `n_features` from 2000 to 50 and re-run. Does the leaky R² get larger or smaller? Explain why the number of candidate features matters.
2. Change `Ridge(alpha=1.0)` to `Ridge(alpha=100.0)` in section 4 and compare cross-validation results. Which metric changed most?
3. MAE and RMSE can rank two models differently. Construct a small example by hand where model A has lower MAE but higher RMSE than model B, and explain what that tells you about their errors.

**Do not** use the test set to decide between variations.

## 9. What's next, and where to get help

- `Part-1A_Classification_ML_Pipeline.ipynb` — the same workflow for a categorical target, plus `ColumnTransformer` for categorical health columns.
- `../handouts/leakage_checklist.md` — the checklist this section demonstrates.
- `../handouts/metric_chooser.md` — choosing between MAE, RMSE, and R².
- `exercises_and_answer_key.md`, `troubleshooting.md`.

## Summary and limits

- [x] Inspected the target's spread before interpreting any error metric.
- [x] Split once, before fitting anything.
- [x] Built preprocessing inside a pipeline.
- [x] Compared against a dummy baseline with cross-validation on training data only.
- [x] Evaluated once on the held-out test set and explained negative R².
- [x] Read residual plots and identified prediction compression toward the mean.
- [x] Demonstrated preprocessing leakage producing apparent skill from pure noise.

**Limits:** this is one internal estimate on one small benchmark dataset with one split. It does not establish causal effects, individual prognosis, fairness, calibration, or performance in another population or time period.